# Q1 Data Audit

Exploratory audit of the supplied Campaign Team Tracking and Coverage Reconciliation datasets. This notebook reads source files without modifying them and records descriptive findings only.

**Scope:** inventory, structure, completeness, validity, and duplicate checks only. No quality thresholds are applied, no source records are removed, and no analytical conclusions are drawn.

**Reproducibility:** input locations are resolved relative to the repository root. By default, the notebook expects the supplied data pack in its retained local location; set the `EHA_Q1_DATA_DIR` environment variable to point to another copy of the same Q1 source folder. The only generated file is the descriptive audit summary in `outputs/tables/`.

## 1. Environment and imports

Load the libraries used for tabular and spatial inspection, then resolve project-relative input and output paths. The path check stops execution with a clear message if the supplied data pack is not available.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import fiona
import geopandas as gpd
import pandas as pd

pd.set_option('display.max_columns', 100)

def field_inventory(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a consistent descriptive inventory without altering source records."""
    return pd.DataFrame({
        'field': frame.columns,
        'data_type': frame.dtypes.astype(str).values,
        'missing_values': frame.isna().sum().values,
    })


PROJECT_NAME = 'Q1_Campaign_Team_Tracking'
cwd = Path.cwd().resolve()
try:
    REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / PROJECT_NAME).exists())
except StopIteration as error:
    raise RuntimeError(
        f'Could not locate the repository root containing {PROJECT_NAME}. Run this notebook from within the repository.'
    ) from error
DATA_ROOT = Path(os.environ.get(
    'EHA_Q1_DATA_DIR',
    REPO_ROOT / 'Technical_asssessment' / 'eHA_Assessment_Data_Pack_v4_CANDIDATE' / 'Part1_Q1_Campaign_Tracking',
))
OUTPUT_PATH = REPO_ROOT / PROJECT_NAME / 'outputs' / 'tables' / 'data_audit_summary.csv'

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'Source data not found at {DATA_ROOT}. Set EHA_Q1_DATA_DIR to the supplied Q1 data folder.'
    )

print(f'Repository root: {REPO_ROOT}')
print(f'Source data: {DATA_ROOT}')

## 2. GPS track inventory

Inventory every supplied GPS CSV before any transformation. The audit records file coverage, team and date identifiers, record counts, schema variants, missing values, coordinate-range exceptions, and timestamp parse failures. Invalid or missing values are counted and retained in the source files.

In [ ]:
track_files = sorted((DATA_ROOT / 'tracks').glob('*.csv'))
if not track_files:
    raise FileNotFoundError('No GPS CSV files found in the tracks directory.')

inventory_rows = []
schema_signatures = {}
for file_path in track_files:
    frame = pd.read_csv(file_path)
    parsed_timestamp = pd.to_datetime(frame['timestamp'], errors='coerce')
    signature = tuple(frame.columns)
    schema_signatures.setdefault(signature, []).append(file_path.name)
    inventory_rows.append({
        'file_name': file_path.name,
        'records': len(frame),
        'team_ids': ', '.join(sorted(frame['team_id'].dropna().astype(str).unique())),
        'dates': ', '.join(sorted(parsed_timestamp.dropna().dt.date.astype(str).unique())),
        'missing_values': int(frame.isna().sum().sum()),
        'invalid_timestamps': int(parsed_timestamp.isna().sum()),
        'longitude_out_of_range': int((~frame['longitude'].between(-180, 180)).sum()),
        'latitude_out_of_range': int((~frame['latitude'].between(-90, 90)).sum()),
    })

gps_inventory = pd.DataFrame(inventory_rows)
gps_inventory.head()

The next cell consolidates file-level observations into campaign-wide counts and displays the observed schemas. It does not standardize, filter, or overwrite any track records.

In [ ]:
gps_totals = {
    'file_count': len(track_files),
    'team_count': gps_inventory['team_ids'].str.split(', ').explode().nunique(),
    'campaign_date_count': gps_inventory['dates'].str.split(', ').explode().nunique(),
    'total_records': int(gps_inventory['records'].sum()),
    'total_missing_values': int(gps_inventory['missing_values'].sum()),
    'invalid_timestamps': int(gps_inventory['invalid_timestamps'].sum()),
    'longitude_out_of_range': int(gps_inventory['longitude_out_of_range'].sum()),
    'latitude_out_of_range': int(gps_inventory['latitude_out_of_range'].sum()),
    'schema_variants': len(schema_signatures),
}

print('GPS track totals')
display(pd.Series(gps_totals))
print('Schema consistency')
for columns, files in schema_signatures.items():
    print(f'{len(files)} file(s): {list(columns)}')

gps_inventory[['team_ids', 'dates']].drop_duplicates().sort_values(['team_ids', 'dates'])

## 3. Settlement masterlist audit

Inspect the planned settlement reference data for identifier uniqueness, coordinate completeness, and administrative distribution. Counts are reported by LGA and ward to expose the structure of the supplied masterlist without changing it.

In [ ]:
settlements = pd.read_csv(DATA_ROOT / 'settlement_masterlist.csv')
settlement_summary = {
    'row_count': len(settlements),
    'field_count': len(settlements.columns),
    'duplicate_settlement_ids': int(settlements['settlement_id'].duplicated().sum()),
    'missing_coordinates': int(settlements[['longitude', 'latitude']].isna().any(axis=1).sum()),
}

display(settlements.head())
display(field_inventory(settlements))
display(pd.Series(settlement_summary))
display(settlements.groupby('lga_name', dropna=False).size().rename('settlement_count').reset_index())
display(settlements.groupby(['lga_name', 'ward_name'], dropna=False).size().rename('settlement_count').reset_index())

## 4. eTally audit

Review the daily e-tally dataset for expected fields, completeness, repeated date/team/settlement combinations, and dose values requiring later investigation. The suspicious-dose check is descriptive: it flags negative, missing, or target-exceeding values without treating them as errors or deleting them.

In [ ]:
etally = pd.read_csv(DATA_ROOT / 'etally_daily.csv')
etally_key = ['campaign_date', 'team_id', 'settlement_id']
etally_duplicates = etally[etally.duplicated(etally_key, keep=False)].sort_values(etally_key)
suspicious_doses = etally[
    etally['doses_administered'].isna()
    | (etally['doses_administered'] < 0)
    | (etally['doses_administered'] > etally['target_population_under5'])
].copy()
etally_summary = {
    'row_count': len(etally),
    'field_count': len(etally.columns),
    'duplicate_key_rows': len(etally_duplicates),
    'missing_values': int(etally.isna().sum().sum()),
    'suspicious_dose_rows': len(suspicious_doses),
}

display(etally.head())
display(field_inventory(etally))
display(pd.Series(etally_summary))
display(etally_duplicates.head())
display(suspicious_doses.head())

## 5. Inaccessible settlements audit

Describe the supplied security-accessibility reference data, including completeness and administrative distribution. This audit does not infer causes of inaccessibility or alter the classification.

In [ ]:
inaccessible = pd.read_csv(DATA_ROOT / 'inaccessible_settlements.csv')
inaccessible_summary = {
    'row_count': len(inaccessible),
    'field_count': len(inaccessible.columns),
    'missing_values': int(inaccessible.isna().sum().sum()),
}

display(inaccessible.head())
display(field_inventory(inaccessible))
display(pd.Series(inaccessible_summary))
display(inaccessible.groupby('lga_name', dropna=False).size().rename('settlement_count').reset_index())
display(inaccessible.groupby(['lga_name', 'ward_name'], dropna=False).size().rename('settlement_count').reset_index())

## 6. Boundary audit

Load each GeoPackage layer and report its name, feature count, coordinate reference system, missing geometries, and geometry-validity status. Geometry exceptions are reported for review; no geometries are repaired or excluded.

In [ ]:
boundary_path = DATA_ROOT / 'boundaries.gpkg'
if not boundary_path.exists():
    raise FileNotFoundError(f'Boundary GeoPackage not found: {boundary_path}')

boundary_rows = []
for layer_name in fiona.listlayers(boundary_path):
    layer = gpd.read_file(boundary_path, layer=layer_name)
    boundary_rows.append({
        'layer': layer_name,
        'feature_count': len(layer),
        'crs': str(layer.crs),
        'invalid_geometries': int((~layer.geometry.is_valid).sum()),
        'missing_geometries': int(layer.geometry.isna().sum()),
    })

boundary_audit = pd.DataFrame(boundary_rows)
display(boundary_audit)

## 7. Audit summary table

The table below consolidates descriptive audit findings into the required output file, `outputs/tables/data_audit_summary.csv`. It does not apply quality thresholds, modify source data, or make analytical decisions.

In [ ]:
audit_summary = pd.DataFrame([
    {'dataset': 'GPS tracks', 'metric': 'File count', 'value': gps_totals['file_count']},
    {'dataset': 'GPS tracks', 'metric': 'Team count', 'value': gps_totals['team_count']},
    {'dataset': 'GPS tracks', 'metric': 'Campaign date count', 'value': gps_totals['campaign_date_count']},
    {'dataset': 'GPS tracks', 'metric': 'Total records', 'value': gps_totals['total_records']},
    {'dataset': 'GPS tracks', 'metric': 'Schema variants', 'value': gps_totals['schema_variants']},
    {'dataset': 'GPS tracks', 'metric': 'Missing values', 'value': gps_totals['total_missing_values']},
    {'dataset': 'GPS tracks', 'metric': 'Invalid timestamps', 'value': gps_totals['invalid_timestamps']},
    {'dataset': 'GPS tracks', 'metric': 'Coordinates outside valid ranges', 'value': gps_totals['longitude_out_of_range'] + gps_totals['latitude_out_of_range']},
    {'dataset': 'Settlement masterlist', 'metric': 'Row count', 'value': settlement_summary['row_count']},
    {'dataset': 'Settlement masterlist', 'metric': 'Duplicate settlement IDs', 'value': settlement_summary['duplicate_settlement_ids']},
    {'dataset': 'Settlement masterlist', 'metric': 'Records with missing coordinates', 'value': settlement_summary['missing_coordinates']},
    {'dataset': 'eTally', 'metric': 'Row count', 'value': etally_summary['row_count']},
    {'dataset': 'eTally', 'metric': 'Duplicate date/team/settlement rows', 'value': etally_summary['duplicate_key_rows']},
    {'dataset': 'eTally', 'metric': 'Missing values', 'value': etally_summary['missing_values']},
    {'dataset': 'eTally', 'metric': 'Potentially suspicious dose rows', 'value': etally_summary['suspicious_dose_rows']},
    {'dataset': 'Inaccessible settlements', 'metric': 'Row count', 'value': inaccessible_summary['row_count']},
    {'dataset': 'Inaccessible settlements', 'metric': 'Missing values', 'value': inaccessible_summary['missing_values']},
])

for row in boundary_audit.itertuples(index=False):
    audit_summary.loc[len(audit_summary)] = {
        'dataset': f'Boundary layer: {row.layer}',
        'metric': 'Feature count',
        'value': row.feature_count,
    }
    audit_summary.loc[len(audit_summary)] = {
        'dataset': f'Boundary layer: {row.layer}',
        'metric': 'Invalid geometries',
        'value': row.invalid_geometries,
    }

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
audit_summary.to_csv(OUTPUT_PATH, index=False)
display(audit_summary)
print(f'Audit summary written to: {OUTPUT_PATH}')

## Audit Findings and Next Steps

Complete this section after running the descriptive audit. It is a record of observations and open questions, not a place to set analytical thresholds or select methods.

### Observed Data Issues

_To be completed from the audit output._

### Possible Impact

_To be assessed after the observed issues are documented._

### Decisions Still Required

_To be recorded in the technical decision log before implementation._

### Follow-up Actions

_To be defined after review of the audit findings._